# ❤️ Heart Disease Prediction — Professional ML Pipeline
**Dataset:** Framingham Heart Study | **Author:** Swathi K S | **Goal:** Predict 10-Year CHD Risk

---
### Pipeline Overview
1. Exploratory Data Analysis (EDA)
2. Smart Missing Value Imputation
3. Class Imbalance Handling with SMOTE
4. Multi-Model Training & Comparison
5. AUC-ROC, Confusion Matrix, F1 Evaluation
6. Feature Importance & SHAP Explainability
7. Model Persistence

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score,
    classification_report, confusion_matrix, roc_curve
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')
print('All libraries loaded successfully ✅')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/framingham.csv')
df.drop(['education'], axis=1, inplace=True)
df.rename(columns={'male': 'gender'}, inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Class Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['TenYearCHD'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(counts, labels=['No CHD', 'CHD Risk'], autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Class Distribution (Severe Imbalance!)', fontsize=14, fontweight='bold')

axes[1].bar(['No CHD (0)', 'CHD Risk (1)'], counts.values, color=colors, edgecolor='white')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=13)
axes[1].set_title('Count of Each Class', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')

plt.suptitle('Target Variable: 10-Year CHD Risk', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Class ratio: {counts[0]/counts[1]:.1f}:1 — significant imbalance, SMOTE needed!')

In [ ]:
# ── Missing Values Heatmap ──
fig, ax = plt.subplots(figsize=(12, 5))
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0]

bars = ax.barh(missing_df.index, missing_df['Percentage'],
               color=sns.color_palette('Reds_r', len(missing_df)))
for bar, pct in zip(bars, missing_df['Percentage']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontweight='bold')
ax.set_xlabel('Missing %', fontsize=12)
ax.set_title('Missing Values per Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ──
fig, ax = plt.subplots(figsize=(13, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Key Risk Factors by CHD Status ──
risk_features = ['age', 'totChol', 'sysBP', 'glucose', 'BMI', 'cigsPerDay']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(risk_features):
    for chd_val, label, color in [(0,'No CHD','#2ecc71'), (1,'CHD Risk','#e74c3c')]:
        axes[i].hist(df[df['TenYearCHD']==chd_val][feat].dropna(),
                     bins=30, alpha=0.6, label=label, color=color, density=True)
    axes[i].set_title(f'{feat} Distribution by CHD Status', fontweight='bold')
    axes[i].legend()
    axes[i].set_ylabel('Density')

plt.suptitle('Key Risk Factor Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/risk_factor_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing — Imputation + SMOTE

In [ ]:
FEATURES = ['age','gender','currentSmoker','cigsPerDay','BPMeds',
            'prevalentStroke','prevalentHyp','diabetes','totChol',
            'sysBP','diaBP','BMI','heartRate','glucose']

X = df[FEATURES]
y = df['TenYearCHD']

# Stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Impute BEFORE SMOTE (critical!)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

# SMOTE — synthetic minority oversampling
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_imp, y_train)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test_imp)

print(f'Train set before SMOTE: {dict(y_train.value_counts())}')
print(f'Train set after SMOTE:  {dict(zip(*np.unique(y_train_sm, return_counts=True)))}')
print(f'Test set (untouched):   {dict(y_test.value_counts())}')
print('\n✅ Preprocessing complete — class imbalance fixed!')

## 5. Multi-Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':             xgb.XGBClassifier(eval_metric='logloss', random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train_sm)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_prob),
        'F1-Score':  f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
    }

results_df = pd.DataFrame(results).T.round(4)
results_df.sort_values('AUC-ROC', ascending=False, inplace=True)
results_df.style.background_gradient(cmap='Greens').format('{:.4f}')

In [ ]:
# ── Visual Model Comparison ──
metrics = ['Accuracy', 'AUC-ROC', 'F1-Score', 'Recall']
x = np.arange(len(results_df))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(results_df.index, rotation=15, ha='right', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.5, label='0.7 threshold')
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── AUC-ROC Curves for All Models ──
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c','#3498db','#2ecc71','#9b59b6','#f39c12']

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1.5, alpha=0.5, label='Random Classifier')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — All Models', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Best Model: Confusion Matrix ──
best_name = results_df.index[0]  # highest AUC-ROC
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_sc)

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: No CHD','Pred: CHD'],
            yticklabels=['True: No CHD','True: CHD'],
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBest Model: {best_name}')
print(classification_report(y_test, y_pred_best, target_names=['No CHD','CHD Risk']))

## 6. Feature Importance & SHAP Explainability

In [ ]:
# ── Random Forest Feature Importance ──
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v > fi.mean() else '#3498db' for v in fi.values]
fi.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(fi.mean(), color='black', linestyle='--', alpha=0.7, label=f'Mean importance')
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Values (Model Explainability) ──
# SHAP shows HOW each feature pushes the prediction up or down
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_sc[:200])  # sample for speed

# For binary classification, shap_values is a list [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values[:,:,1]

plt.figure(figsize=(12, 7))
shap.summary_plot(sv, X_test_sc[:200], feature_names=FEATURES,
                  plot_type='bar', show=False, color='#e74c3c')
plt.title('SHAP Feature Importance — Mean |SHAP value|', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n💡 SHAP explains WHY the model made each prediction — key for medical AI trust!')

In [ ]:
# ── SHAP Beeswarm — impact direction ──
plt.figure(figsize=(12, 8))
shap.summary_plot(sv, X_test_sc[:200], feature_names=FEATURES, show=False)
plt.title('SHAP Summary — Feature Impact on CHD Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cross-Validation & Model Persistence

In [ ]:
# ── 5-Fold Stratified Cross-Validation on best model ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_sc, y_train_sm, cv=cv, scoring='roc_auc')

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, 6), cv_scores, color='#3498db', alpha=0.8, edgecolor='white')
ax.axhline(cv_scores.mean(), color='#e74c3c', linestyle='--', lw=2,
           label=f'Mean AUC = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('AUC-ROC Score', fontsize=12)
ax.set_title(f'5-Fold Cross-Validation — {best_name}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../reports/figures/cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nCV AUC-ROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Save models for deployment ──
pickle.dump(best_model, open('../models/best_model.pkl', 'wb'))
pickle.dump(scaler,     open('../models/scaler.pkl', 'wb'))
pickle.dump(imputer,    open('../models/imputer.pkl', 'wb'))

print('Models saved to /models/')
print(f'\n🏆 Best Model:    {best_name}')
print(f'🎯 AUC-ROC:       {results_df.loc[best_name, "AUC-ROC"]:.4f}')
print(f'📊 F1-Score:      {results_df.loc[best_name, "F1-Score"]:.4f}')
print(f'🔁 CV AUC (mean): {cv_scores.mean():.4f}')

## 8. Key Insights
- **Age** and **systolic blood pressure (sysBP)** are the strongest predictors of CHD risk
- **Glucose** and **cholesterol** also show meaningful separability between classes
- The original dataset had a severe **5.6:1 class imbalance** — SMOTE was essential
- **AUC-ROC** is the right metric here (not accuracy) due to class imbalance
- SHAP values reveal the model is clinically interpretable — critical for medical AI

## 9. Next Steps
- Deploy via Streamlit web app for interactive predictions
- Try ensemble stacking for further AUC improvement
- Collect more minority class samples if available
- Integrate with a MySQL DB to log predictions